<a href="https://colab.research.google.com/github/TRPG-masterbot/data/blob/main/Kim/sce_rag_v5_%ED%86%B5%ED%95%A9%EC%9A%A9.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🟥 시나리오 RAG (잠긴 종) — v5 통합용

통합 노트북에 붙여 쓰는 **기능 셀만** 모은 버전입니다. 골든셋·평가·자유 입력 시험(`input()`) 셀은 뺐습니다. (시험은 `sce_rag_v5.ipynb`)

**v5 변경점** — 검색 필터가 '가 본 방'만 보고 '지금 서 있는 방'은 안 봐서, 지금 방에 없는 인물이 서술에 나오는 문제가 있었다.
state에 현재 방(`room`)을 추가해 **현재 방의 카드만 검색·서술**하게 했다 (##6 `search`, ##11 `narrate_scene`).

| 셀 | 하는 일 |
|---|---|
| ##0 | import |
| ##1~##2 | 카드 읽기, 공개 카드(검색 대상)만 남기기 |
| ##3 | 검색 가능 여부 규칙 `can_search` |
| ##4~##5 | 카드 → 청크 → 임베딩 → Chroma |
| ##6 | **`search(question, state, k)`** — 상태 필터 + 현재 방 필터 |
| ##10 | `ask_llm` (Ollama 호출) |
| ##11 | **`narrate_scene(text, state, history)`** — 검색 + 서술 + 규칙 검사 |

**밖에서 부르는 함수** (통합 코드가 쓰는 것)
- `search(question, state, k=3)` → `[(카드 id, 유사도)]`
- `narrate_scene(text, state, history)` → `(서술, 카드, {"tries", "problems"})`

**state 모양**: `{"scene": 3, "flags": set, "visited": set, "cls": "fighter", "room": "cloister_mid"}`
`room`은 없어도 동작하지만(방 제한 없음), 있어야 현재 방 제한이 걸린다.

**통합 코드에서 같이 고칠 곳 (팀 파일이라 직접 고치지 않음)**
입력 파트의 `_scenario_game_state(state)`가 검색용 상태를 다시 만들면서 `room`을 버린다. `return result` 앞에 추가:
```python
    if isinstance(state.get("room"), str) and state["room"]:
        result["room"] = state["room"]          # 현재 방을 검색 필터까지 넘긴다
```

In [ ]:
# ===== ##0 import =====
# 역할: 시나리오 RAG에 필요한 도구를 불러온다. (설치는 통합 노트북의 0. 공통에서 함)
import json
import re                                    # [v5] 서술 검사용 (통합 노트북에는 이미 있음)
import requests
from collections import Counter

import chromadb
from sentence_transformers import SentenceTransformer

In [ ]:
## 1. cards.json 가져오기
# 역할: 시나리오 카드 166장과 방 이름표를 읽는다.
CARDS_URL = "https://raw.githubusercontent.com/TRPG-masterbot/data/main/BAE/src/game/data/cards.json"   # [결합] 게임 패키지 카드(출력 파트와 같은 파일)로 통일
response = requests.get(CARDS_URL, timeout=60)
response.raise_for_status()
with open("cards.json", "wb") as f:
    f.write(response.content)

with open("cards.json", "r", encoding="utf-8") as f:
    data = json.load(f)

cards = data["cards"]      # 카드 166장
rooms = data["rooms"]      # 방 id -> 한글 이름표 (27개)

In [ ]:
## 2. 공개용 / GM 전용 나누기
# 역할: player_text가 있는 카드만 검색 대상(search_cards)으로 삼는다. GM 전용 카드(player_text가 빈 카드)는 제외해 비밀이 섞이지 않게 한다.
search_cards = [c for c in cards if c["player_text"]]

kinds = Counter(c["kind"] for c in search_cards)
for banned in ["secret", "check", "monster"]:      # 비밀 종류 카드가 섞였는지 확인
    assert banned not in kinds, banned

In [ ]:
## 3. 현재 상황에서 검색 가능한 카드만 거르기
# 역할: "지금 이 플레이어에게 보여도 되는 카드인가?" (장면, 플래그, 가 본 방, 직업)

def can_search(card, state): # 필터
    if state["scene"] < card["reveal_after"]:
        return False        # 아직 공개 장면이 아님
    if card["req_flag"] and card["req_flag"] not in state["flags"]:
        return False        # 필요한 플래그가 없음 (req_flag가 빈 글자면 이 검사는 건너뜀)
    if card["room"] and card["room"] not in state["visited"]:
        return False        # 아직 가 보지 않은 방
    if card["owner"] != "all" and card["owner"] != state["cls"]:
        return False        # 다른 직업 전용 카드
    return True

In [ ]:
## 4. 청크 만들기
# 역할: 카드 1장 = 청크 1개. "[종류] 방이름" 머리말 + player_text (GM 문장은 넣지 않는다).

card_by_id = {c["id"]: c for c in cards}      # id로 카드 원본을 바로 찾기 위한 사전

def make_chunk_text(card):
    room_name = rooms.get(card["room"], "")   # 엔딩 카드 9장은 room이 비어 있어서 .get 사용
    header = f"[{card['label']}]"
    if room_name:
        header += f" {room_name}"
    return header + "\n" + card["player_text"]

chunks = [{"id": c["id"], "text": make_chunk_text(c)} for c in search_cards]

for c, ch in zip(search_cards, chunks):       # GM 문장이 청크에 섞이지 않았는지 확인
    if c["gm_only_text"]:
        assert c["gm_only_text"] not in ch["text"], c["id"]

In [ ]:
## 5. 임베딩 + Chroma 저장
# 역할: 청크를 벡터로 바꿔 Chroma(메모리)에 저장한다. 검색은 이 벡터로 한다.

model = SentenceTransformer("intfloat/multilingual-e5-small") # 가벼운 기본모델

texts = ["passage: " + ch["text"] for ch in chunks]
vectors = model.encode(texts, normalize_embeddings=True, show_progress_bar=False)

client = chromadb.Client()      # 메모리에만 저장 (런타임을 끄면 사라짐)
collection = client.get_or_create_collection(
    name="scenario_cards",
    metadata={"hnsw:space": "cosine"},
)
collection.upsert(
    ids=[ch["id"] for ch in chunks],
    documents=[ch["text"] for ch in chunks],
    embeddings=vectors.tolist(),
)

In [ ]:
## 6. 상태를 반영한 검색 함수
# 역할: 질문과 비슷한 카드를 찾되, ① 지금 볼 수 있는 카드만(can_search) ② [v5] 지금 서 있는 방의 카드만 남긴다.

ROOM_ONLY = True    # [v5] True: 현재 방(state["room"])의 카드와 방 없는 카드(엔딩)만 검색 / False: 가 본 방 전체

def search(question, state, k=3, use_filter=True):
    q_vec = model.encode("query: " + question, normalize_embeddings=True)
    res = collection.query(query_embeddings=[q_vec.tolist()], n_results=len(chunks))
        # 질문을 벡터로 바꿔 카드 전체를 유사한 순서로 가져옴

    room_now = state.get("room")                  # [v5] 지금 서 있는 방 (없으면 None -> 방 제한을 걸지 않음)
    # 비슷한 순서대로 훑으면서, 지금 볼 수 있는 카드만 남겨 앞에서 k개를 돌려준다
    hits = []
    for cid, dist in zip(res["ids"][0], res["distances"][0]):
        card = card_by_id[cid]
        if use_filter and not can_search(card, state):
            continue                              # 지금 볼 수 없는 카드는 건너뜀
        if use_filter and ROOM_ONLY and room_now and card["room"] and card["room"] != room_now:
            continue                              # [v5] 다른 방의 카드는 건너뜀 (방이 없는 엔딩 카드는 통과)
        hits.append((cid, round(1 - dist, 3)))    # (카드 id, 유사도)
    return hits[:k]

In [ ]:
# ===== ##10 모델 호출 =====
# 역할: Ollama 모델에 대화를 보내고 답을 받는다. (Ollama 설치·모델 받기는 통합 노트북의 0. 공통에서 함)
# 통합 노트북에서는 0. 공통의 OLLAMA_URL, GEN_MODEL, NUM_CTX를 그대로 쓴다. 없을 때만 아래 기본값.
OLLAMA_URL = globals().get("OLLAMA_URL", "http://localhost:11434")
GEN_MODEL = globals().get("GEN_MODEL", "qwen3.5:9b")
NUM_CTX = globals().get("NUM_CTX", 8192)

def ask_llm(messages, max_new_tokens=300, seed=42, temperature=0.0):
    res = requests.post(f"{OLLAMA_URL}/api/chat", timeout=600, json={
        "model": GEN_MODEL, "messages": messages, "stream": False,
        "think": False,                               # [결합] qwen3.5 생각 모드 끄기
        "options": {"temperature": temperature,      # 0 = 같은 입력 -> 같은 출력 (재시도할 때는 올린다)
                    "seed": seed, "num_ctx": NUM_CTX, "num_predict": max_new_tokens}})
    res.raise_for_status()
    return res.json()["message"]["content"].strip()

In [ ]:
# ===== ##11 서술 규칙 프롬프트 + 검사 + 한 턴 함수 =====
# 역할: 검색한 카드만 근거로 모델이 장면을 서술하게 하고, 규칙을 어기면 다시 쓰게 한다. [v5] 장소는 현재 방 기준.
GM_SYSTEM = ("You are the game master of a Korean single-player D&D adventure. Write every reply in natural Korean (Hangul). "
             "Never use Chinese characters, Japanese, or any other language.")

GM_TEMPLATE = """당신은 1인용 TRPG의 게임 마스터입니다. 아래 규칙을 지키세요.
1. [참고 자료]에 있는 내용만 사용해 지금 장면을 묘사하세요. 자료에 없는 인물, 장소, 사건, 물건을 지어내지 마세요.
2. 플레이어를 '당신'이라고 부르세요. 당신의 행동, 생각, 감정, 대사를 대신 정하지 마세요.
3. 행동은 시도하는 모습과 눈에 보이는 반응까지만 쓰세요. 물건을 받거나 얻는 것, 길이 열리는 것, 누가 쓰러지거나 동의하는 것 같은 결과는 쓰지 마세요. 결과는 게임이 따로 정합니다.
4. 입력에 답할 수 있는 내용이 [참고 자료]에 없으면, 지금은 새로 알 수 있는 것이 없다고 짧게 쓰세요.
5. NPC는 참고 자료에 적힌 성격대로 짧게 말하게 하되, 자료에 없는 사실을 말하게 하지 마세요.
6. 플레이어가 지난 일을 물으면 [최근 기록]에 있는 내용으로 짧게 알려 주세요. 거기 없으면 기억나지 않는다고 하세요.
7. 2~4문장으로 쓰세요. HP, DC 같은 게임 수치와 출처는 쓰지 마세요. 마지막에 "어떻게 하시겠어요?" 같은 질문은 쓰지 마세요.
8. [장소]에 없는 인물이나 장소는 등장시키지 마세요. 지금 이 방에서 보이는 것만 쓰세요.

[장소] {room}

[참고 자료]
{docs}

[최근 기록]
{history}

[플레이어] {action}
[GM]"""

# 서술을 검사하는 규칙 (걸리면 문제를 알려 주고 다시 쓰게 한다)
BROKEN = re.compile(r"[^가-힣ㄱ-ㆎ\x00-\x7f\s…·“”‘’—–]")        # 한글·영문·기호 말고 다른 글자 (한자 등)
LEAK = re.compile(r"\b(HP|DC|CR|SRD)\b|p\.\s*\d+|scn-")                          # 게임 수치·출처 노출
OUTCOME = re.compile(r"(건네|건넨|넘겨|내어\s*준|내준|받았|받는다|획득|쥐여|꺼내\s*준|얻었|얻는다|열렸|쓰러졌|쓰러진다)")   # 결과를 확정하는 말
NO_NEWS = "지금은 이 행동을 할 수 없을 거 같다. 다른 행동을 해 보자."

def narration_problems(body):
    p = []
    if BROKEN.search(body): p.append("깨진 글자")
    if LEAK.search(body): p.append("게임 수치·출처 노출")
    if len(body) < 20: p.append("너무 짧음")
    if re.search(r"플레이어|주인공", body): p.append("호칭")
    if OUTCOME.search(body): p.append("행동의 결과를 확정함")
    return p

def clean_body(body):
    # 끝까지 문제가 남았을 때: 문제 있는 문장을 빼고, 남는 게 없으면 한 줄 안내
    body = re.sub(r"플레이어|주인공", "당신", body)
    sents = re.split(r"(?<=[.!?…])\s+|\n+", body)
    kept = " ".join(s for s in sents if s and not (BROKEN.search(s) or LEAK.search(s) or OUTCOME.search(s))).strip()
    return kept if len(kept) >= 20 else NO_NEWS

def recent_records(history, n=6):
    return "\n".join(f"플레이어: {a} / GM: {b}" for a, b in history[-n:]) or "(없음)"

def narrate_scene(text, state, history, k=2, tries=3):
    hits = search(text, state, k=k)                                    # 상태 필터와 현재 방 필터가 적용된 시나리오 검색
    if not hits:                                                       # 볼 수 있는 카드가 없으면 모델을 부르지 않는다
        return NO_NEWS, hits, {"tries": 0, "problems": []}
    docs = [card_by_id[cid]["player_text"] for cid, _ in hits]          # 플레이어용 글만 모델에게 준다
    room_id = state.get("room")                                        # [v5] 지금 서 있는 방
    if room_id:
        room = rooms.get(room_id, room_id)                             # [v5] 실제 현재 방의 이름 (검색된 카드 기준이 아님)
    else:                                                              # 방을 모르면 가장 위 카드의 방
        rooms_here = [rooms.get(card_by_id[cid]["room"], "") for cid, _ in hits]
        room = next((r for r in rooms_here if r), "(알 수 없음)")
    prompt = GM_TEMPLATE.format(room=room, docs="\n\n".join(f"- {d}" for d in docs) or "(없음)",
                                history=recent_records(history), action=text)
    messages = [{"role": "system", "content": GM_SYSTEM}, {"role": "user", "content": prompt}]
    body, probs = "", []
    for t in range(tries):
        body = ask_llm(messages, max_new_tokens=250, seed=42 + t, temperature=0.7 if t else 0.3)
        probs = narration_problems(body)
        if not probs:
            break
        messages = messages[:2] + [{"role": "assistant", "content": body},
                                   {"role": "user", "content": "다음 문제가 있어. 고쳐서 다시 써 줘: " + "; ".join(probs)}]
    if probs:
        body = clean_body(body)
    body = re.split(r"\s*(?:(?:어떻게|무엇을|뭘)\s*(?:계속|대응)?\s*하시겠|\[게임 진행)", body)[0].rstrip()   # 끝의 되묻기와 군더더기 제거
    return body, hits, {"tries": t + 1, "problems": probs}